# S&P 500 Options: LSTM

This notebook fits the declared LSTM member of the sequence population snapshotted by
`09_deep_learning`. Chronological windows, validation gaps, checkpoints, and prediction
eligibility are resolved through the shared sequence boundary.

Prerequisite: `09_deep_learning` must create the complete official sequence population.

In [ ]:
"""Fit the declared S&P 500 options LSTM request."""

import polars as pl

from case_studies.sp500_options.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_subset,
    run_resolved_model_requests,
)

In [ ]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
DEVICE: str = ""

POPULATION_NAME: str = ""

### The device the population was fitted on

A network trained on a GPU and the same network trained on a CPU accumulate their sums in a
different order and reach different weights, so the device is part of what the fitted model is
and sits inside the training identity rather than beside it. `PUBLISHED_DEVICE` is the device
this population was fitted on, and pinning it is what makes the notebook fit the same thing
wherever it runs. On a machine with no NVIDIA card the run stops here rather than quietly
training something else: set `DEVICE="cpu"` and pass a `POPULATION_NAME` to fit the same
requests there, under a name of their own. `08_tabular_dl` pins its device the same way.

In [ ]:
PUBLISHED_DEVICE = "cuda"
CANONICAL_POPULATION_NAME = "sp500-options-sequence-validation-v1"

device = DEVICE or PUBLISHED_DEVICE
population_name = POPULATION_NAME or CANONICAL_POPULATION_NAME
if device != PUBLISHED_DEVICE and population_name == CANONICAL_POPULATION_NAME:
    raise ValueError(
        f"this run fits on {device!r}, not the published {PUBLISHED_DEVICE!r}, so its "
        f"identities are not the ones {CANONICAL_POPULATION_NAME!r} holds; pass "
        f"POPULATION_NAME to give them a population of their own"
    )
print(f"training device: {device}")

## Declared request

In [ ]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
requests = model_request_catalog(
    "deep_learning",
    labels=ALL_LABELS,
    config_names=("lstm_h64",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_model_plan(resolved)

## Execute and validate

The shared sequence runner owns chronological window construction, fold fitting, fitted-state
reload, checkpoint publication, restart, and exact eligible-key validation.

In [ ]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_subset(
        study,
        resolved,
        population=population_name,
    )
else:
    if not WORKSPACE or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

In [ ]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("LSTM execution returned a partial checkpoint")
catalog

The complete LSTM checkpoint population is ready for model analysis and backtesting. This
notebook does not compare it with another family or choose a checkpoint.